# Module 12 — Bayesian Black-Box Optimisation
## Week 1 Submission

**Student:** Divya Bhanusri

**Strategy:** Gaussian Process (GP) + Upper Confidence Bound (UCB) with adaptive beta values

| Function | Dimensions | Beta | Reason |
|---|---|---|---|
| F1 | 2D | 2.5 | Radiation detection — near zero outputs, explore more |
| F2 | 2D | 2.5 | Noisy ML model — complex landscape, explore more |
| F3 | 3D | 1.5 | Drug discovery — moderate complexity, balanced |
| F4 | 4D | 2.5 | Warehouse placement — all negative, explore more |
| F5 | 4D | 1.0 | Chemical yield — unimodal, strong signal, exploit more |
| F6 | 5D | 1.5 | Cake recipe — moderate complexity, balanced |
| F7 | 6D | 1.5 | ML hyperparameters — moderate complexity, balanced |
| F8 | 8D | 1.0 | 8D optimisation — strong signal, exploit more |


## Cell 1 — Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully!')

## Cell 2 — Load All Data

In [ ]:
# Change this path to where your data is stored
base_path = './initial_data/'

# Beta values per function
betas = {
    1: 2.5,  # Noisy/weak signal — explore more
    2: 2.5,  # Noisy/weak signal — explore more
    3: 1.5,  # Moderate complexity — balanced
    4: 2.5,  # All negative — explore more
    5: 1.0,  # Strong signal — exploit more
    6: 1.5,  # Moderate complexity — balanced
    7: 1.5,  # Moderate complexity — balanced
    8: 1.0,  # Strong signal — exploit more
}

descriptions = {
    1: 'Radiation Detection',
    2: 'Noisy ML Model',
    3: 'Drug Discovery',
    4: 'Warehouse Placement',
    5: 'Chemical Yield',
    6: 'Cake Recipe',
    7: 'ML Hyperparameters',
    8: '8D Optimisation'
}

# Load all data
data = {}
for i in range(1, 9):
    X = np.load(f'{base_path}function_{i}/initial_inputs.npy')
    y = np.load(f'{base_path}function_{i}/initial_outputs.npy')
    data[i] = {'X': X, 'y': y}
    print(f'Function {i} ({descriptions[i]}) — Shape: {X.shape} | Best y: {y.max():.4f}')

## Cell 3 — Plot 1: All 8 Functions Overview

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i in range(1, 9):
    X = data[i]['X']
    y = data[i]['y']
    beta = betas[i]

    ax = axes[i-1]
    # Green = best point, Blue = others
    colors = ['green' if val == y.max() else 'steelblue' for val in y]
    ax.bar(range(len(y)), y, color=colors, alpha=0.8, edgecolor='black', lw=0.5)
    ax.axhline(y=y.max(), color='red', linestyle='--', lw=1.5,
               label=f'Best: {y.max():.3f}')
    ax.set_title(f'F{i}: {descriptions[i]}\n({X.shape[1]}D | beta={beta})',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Data Point Index', fontsize=8)
    ax.set_ylabel('y value', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Week 1 — All 8 Functions: Initial Data Overview\n(Green = Best observed point)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('All_Functions_Overview.png', dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved!')

## Cell 4 — GP + UCB Strategy

**GP (Gaussian Process):** Learns from past data and predicts future outputs

**UCB Formula:**
$$UCB = \mu + \beta \times \sigma$$

- $\mu$ = GP predicted mean (exploitation)
- $\sigma$ = GP uncertainty (exploration)  
- $\beta$ = controls explore vs exploit balance

In [ ]:
def suggest_next_query(X, y, beta=1.5, n_candidates=100000, random_state=42):
    """
    Suggest next best query point using GP + UCB.
    
    Parameters:
    -----------
    X            : input data points
    y            : output values
    beta         : exploration parameter (higher = more exploration)
    n_candidates : number of random candidate points
    
    Returns:
    --------
    best_x    : suggested next query point
    formatted : formatted string for portal submission
    mu        : GP mean predictions
    sigma     : GP uncertainty predictions
    candidates: all candidate points evaluated
    """
    np.random.seed(random_state)
    dim = X.shape[1]

    # Train GP
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        normalize_y=True
    )
    gp.fit(X, y)

    # Generate random candidates
    candidates = np.random.uniform(0, 1, (n_candidates, dim))

    # Predict
    mu, sigma = gp.predict(candidates, return_std=True)

    # UCB
    ucb = mu + beta * sigma
    best_x = candidates[np.argmax(ucb)]
    formatted = '-'.join([f'{v:.6f}' for v in best_x])

    return best_x, formatted, mu, sigma, candidates, gp

print('Function defined!')

## Cell 5 — Generate Query Points for All 8 Functions

In [ ]:
results = {}

print('=' * 65)
print('  QUERY POINTS TO SUBMIT IN PORTAL')
print('=' * 65)

for i in range(1, 9):
    X = data[i]['X']
    y = data[i]['y']
    beta = betas[i]

    best_x, formatted, mu, sigma, candidates, gp = suggest_next_query(X, y, beta=beta)
    results[i] = {
        'best_x': best_x, 'formatted': formatted,
        'mu': mu, 'sigma': sigma,
        'candidates': candidates, 'gp': gp
    }

    print(f'\nFunction {i} ({descriptions[i]}) | {X.shape[1]}D | Beta={beta}')
    print(f'  Current best y  : {y.max():.4f}')
    print(f'  Submit this     : {formatted}')

print('\n' + '=' * 65)

## Cell 6 — Plot 2: Heatmaps for F1 and F2 (2D Functions)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, func_num in enumerate([1, 2]):
    X = data[func_num]['X']
    y = data[func_num]['y']
    beta = betas[func_num]
    gp = results[func_num]['gp']
    best_x = results[func_num]['best_x']

    # 50x50 grid for heatmap
    x1 = np.linspace(0, 1, 50)
    x2 = np.linspace(0, 1, 50)
    XX1, XX2 = np.meshgrid(x1, x2)
    X_grid = np.column_stack([XX1.ravel(), XX2.ravel()])
    mu_grid, sigma_grid = gp.predict(X_grid, return_std=True)
    ucb_grid = mu_grid + beta * sigma_grid

    # GP Mean heatmap
    ax1 = axes[idx][0]
    im1 = ax1.imshow(mu_grid.reshape(50, 50), origin='lower',
                     extent=[0,1,0,1], aspect='auto', cmap='viridis')
    plt.colorbar(im1, ax=ax1, label='GP Mean')
    ax1.scatter(X[:,0], X[:,1], c='r', marker='x', s=100, lw=2,
                label='Observed data', zorder=5)
    ax1.scatter(best_x[0], best_x[1], c='gold', marker='*', s=400,
                edgecolors='black', zorder=6, label='Next query')
    ax1.set_title(f'F{func_num}: GP Posterior Mean\n({descriptions[func_num]})',
                  fontweight='bold')
    ax1.set_xlabel('x1'); ax1.set_ylabel('x2')
    ax1.legend(fontsize=8)

    # UCB surface
    ax2 = axes[idx][1]
    im2 = ax2.imshow(ucb_grid.reshape(50, 50), origin='lower',
                     extent=[0,1,0,1], aspect='auto', cmap='plasma')
    plt.colorbar(im2, ax=ax2, label=f'UCB (beta={beta})')
    ax2.scatter(X[:,0], X[:,1], c='r', marker='x', s=100, lw=2,
                label='Observed data', zorder=5)
    ax2.scatter(best_x[0], best_x[1], c='gold', marker='*', s=400,
                edgecolors='black', zorder=6,
                label=f'Next: [{best_x[0]:.3f}, {best_x[1]:.3f}]')
    ax2.set_title(f'F{func_num}: UCB Surface (beta={beta})\n({descriptions[func_num]})',
                  fontweight='bold')
    ax2.set_xlabel('x1'); ax2.set_ylabel('x2')
    ax2.legend(fontsize=8)

plt.suptitle('Week 1 — Functions 1 & 2: GP Mean + UCB Surface\n(Gold star = Next query point)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('F1_F2_heatmaps.png', dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved!')

## Cell 7 — Plot 3: UCB Exploration Space (F3 to F8)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, func_num in enumerate([3, 4, 5, 6, 7, 8]):
    X = data[func_num]['X']
    y = data[func_num]['y']
    beta = betas[func_num]
    mu = results[func_num]['mu']
    sigma = results[func_num]['sigma']
    ucb = mu + beta * sigma
    best_idx = np.argmax(ucb)

    ax = axes[idx]
    sc = ax.scatter(mu, sigma, c=ucb, cmap='plasma', alpha=0.3, s=5)
    plt.colorbar(sc, ax=ax, label='UCB score')
    ax.scatter(mu[best_idx], sigma[best_idx], c='red', marker='*',
               s=300, zorder=5, label='Selected next query')
    ax.set_xlabel('GP Mean (mu)', fontsize=9)
    ax.set_ylabel('GP Std (sigma)', fontsize=9)
    ax.set_title(
        f'F{func_num}: {descriptions[func_num]}\n'
        f'({X.shape[1]}D | beta={beta} | Best y={y.max():.3f})',
        fontsize=9, fontweight='bold'
    )
    ax.legend(fontsize=8)

plt.suptitle('Week 1 — Functions 3-8: UCB Exploration Space\n'
             '(Color = UCB score | Red star = Next query point)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('F3_F8_UCB_space.png', dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved!')

## Cell 8 — Final Summary Table

In [ ]:
print('FINAL SUBMISSION SUMMARY — Week 1')
print('=' * 75)
print(f'{"Fn":<4} {"Description":<22} {"Dim":<5} {"Beta":<6} {"Best y":<14} {"Submit This"}')
print('-' * 75)

for i in range(1, 9):
    X = data[i]['X']
    y = data[i]['y']
    formatted = results[i]['formatted']
    print(f'F{i:<3} {descriptions[i]:<22} {X.shape[1]:<5} {betas[i]:<6} {y.max():<14.4f} {formatted}')

print('=' * 75)
print('\nStrategy  : GP + UCB with adaptive beta values')
print('Kernel    : Matern(nu=2.5)')
print('Candidates: 100,000 random points per function')

## Cell 9 — Next Week: Update with New Data

Portal nundi new y values vasthaayi — aa time ee cell use cheyyi!

In [ ]:
def update_and_suggest(func_num, submitted_x, new_y):
    """
    Update dataset with new result and suggest next query.
    
    Parameters:
    -----------
    func_num    : function number (1-8)
    submitted_x : x values you submitted (list)
    new_y       : y value returned by portal (float)
    """
    X = data[func_num]['X']
    y = data[func_num]['y']

    # Add new observation
    X_new = np.vstack([X, submitted_x])
    y_new = np.append(y, new_y)

    # Update stored data
    data[func_num]['X'] = X_new
    data[func_num]['y'] = y_new

    # Suggest next
    beta = betas[func_num]
    best_x, formatted, _, _, _, _ = suggest_next_query(X_new, y_new, beta=beta)

    print(f'Function {func_num} updated!')
    print(f'  Total data points : {len(y_new)}')
    print(f'  New y received    : {new_y:.4f}')
    print(f'  Best y so far     : {y_new.max():.4f}')
    print(f'  Next query        : {formatted}')
    return formatted

# EXAMPLE — Next week portal result vasthundi aa time ila use cheyyi:
# update_and_suggest(
#     func_num    = 1,
#     submitted_x = [0.020584, 0.969910],  # nuvvu submit chesindi
#     new_y       = 0.5                     # portal ichina y value
# )

print('Ready for next week updates!')

## Reflection

### Strategy Used
Gaussian Process (GP) with Matern(nu=2.5) kernel, combined with Upper Confidence Bound (UCB) acquisition function. Beta values were adapted per function based on their characteristics.

### Most Challenging Function
Function 1 — all initial outputs were near zero, giving the GP almost no signal to learn from.

### Next Steps
- Update beta values based on new results each week
- Try WhiteKernel for noisy functions (F1, F2)
- Try RBF kernel for smooth functions (F5)
- Gradually shift from exploration to exploitation as data grows